In [1]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)


def find_repo_root(start: Path) -> Path:
    """Find the repo root by walking upwards until PremierLeague_data exists."""
    start = start.resolve()
    for parent in [start, *start.parents]:
        if (parent / 'PremierLeague_data').exists():
            return parent
    raise FileNotFoundError(
        "Could not locate repo root containing 'PremierLeague_data'. "
        f"Started from: {start}"
    )


REPO_ROOT = find_repo_root(Path.cwd())
DATA_ROOT = REPO_ROOT / 'PremierLeague_data/2024/processed'
ACTIONS_COMPLETE = DATA_ROOT / 'actions_complete.parquet'
ACTIONS_CLEAN = DATA_ROOT / 'actions_clean.parquet'

assert ACTIONS_COMPLETE.exists(), f'Missing {ACTIONS_COMPLETE}'
assert ACTIONS_CLEAN.exists(), f'Missing {ACTIONS_CLEAN}'

print('Repo root:', REPO_ROOT)
print('OK - found:', ACTIONS_COMPLETE)
print('OK - found:', ACTIONS_CLEAN)


Repo root: /home/macaco3001/Projectos/Twelve/twelve-deep-learning
OK - found: /home/macaco3001/Projectos/Twelve/twelve-deep-learning/PremierLeague_data/2024/processed/actions_complete.parquet
OK - found: /home/macaco3001/Projectos/Twelve/twelve-deep-learning/PremierLeague_data/2024/processed/actions_clean.parquet


In [2]:
# Load a manageable sample of matches for quick iteration.
# You can increase N_MATCHES once the logic looks correct.

N_MATCHES = 3

pf = pq.ParquetFile(ACTIONS_COMPLETE)
all_cols = [pf.schema.column(i).name for i in range(pf.metadata.num_columns)]

use_cols = [
    'match_id','period','frame_start','frame_end',
    'event_id','team_id','player_id',
    'event_type','end_type','pass_outcome',
    'x_start','y_start','x_end','y_end',
    'team_in_possession_phase_type','phase_index',
]
use_cols = [c for c in use_cols if c in all_cols]

# Determine a few match_ids to sample without reading the whole file into memory
match_ids = pd.read_parquet(ACTIONS_COMPLETE, columns=['match_id']).head(20000)['match_id'].drop_duplicates().head(N_MATCHES).tolist()
print('Sample match_ids:', match_ids)

df = pd.read_parquet(ACTIONS_COMPLETE, columns=use_cols)
df = df[df['match_id'].isin(match_ids)].copy()

print('Loaded sample actions_complete:', df.shape)
display(df.head())

# Sanity: these are supposed to be player_possession-level events
print('event_type counts (sample):')
display(df['event_type'].value_counts(dropna=False))
print('end_type counts (sample):')
display(df['end_type'].value_counts(dropna=False))

Sample match_ids: [1650385, 1650961, 1651700]
Loaded sample actions_complete: (3918, 16)


,match_id,period,frame_start,frame_end,event_id,team_id,player_id,event_type,end_type,pass_outcome,x_start,y_start,x_end,y_end,team_in_possession_phase_type,phase_index
0,1650385,1,52,52,8_0,48,11438,player_possession,pass,successful,0.37,-0.51,0.37,-0.51,create,0
1,1650385,1,62,71,8_1,48,13068,player_possession,pass,successful,-9.60,-0.26,-9.72,-1.24,create,0
2,1650385,1,87,124,8_2,48,4504,player_possession,pass,successful,-25.30,-6.43,-22.56,-20.46,create,0
3,1650385,1,160,182,8_4,31,70672,player_possession,pass,successful,-8.05,23.63,-13.67,32.22,chaotic,1
4,1650385,1,222,222,8_6,31,1674,player_possession,pass,offside,-9.24,18.66,-9.24,18.66,chaotic,1


event_type counts (sample):


event_type
player_possession    3918
Name: count, dtype: int64

end_type counts (sample):


end_type
pass               3669
possession_loss     109
shot                 99
foul_suffered        29
unknown               6
clearance             6
Name: count, dtype: int64

In [8]:
# Add normalized coordinates from actions_clean (these are the ones used in the project pipeline).
# NOTE: (match_id, event_id) is not guaranteed unique across the full season tables.
# In the processed data, (match_id, event_id, action_id) is the reliable key.

# Ensure we have action_id on df (kernel may already have it from a previous run)
if 'action_id' not in df.columns:
    key_cols = ['match_id','event_id','period','frame_start','frame_end','team_id','player_id']
    extra_cols = key_cols + ['action_id']

    df_action_id = pd.read_parquet(ACTIONS_COMPLETE, columns=extra_cols)
    df_action_id = df_action_id[df_action_id['match_id'].isin(match_ids)].copy()
    df_action_id = df_action_id.drop_duplicates(extra_cols)

    df = df.merge(df_action_id, on=key_cols, how='left')

# If merge created suffixed columns, reconcile them
if 'action_id' not in df.columns and ('action_id_x' in df.columns or 'action_id_y' in df.columns):
    a = df['action_id_x'] if 'action_id_x' in df.columns else np.nan
    b = df['action_id_y'] if 'action_id_y' in df.columns else np.nan
    df['action_id'] = a.fillna(b) if hasattr(a, 'fillna') else b

# Also handle the case where merge created BOTH action_id_x and action_id_y in addition to action_id
# (keep action_id, then drop the suffixed duplicates)
for c in ['action_id_x', 'action_id_y']:
    if c in df.columns:
        df = df.drop(columns=[c])

if df['action_id'].isna().any():
    raise ValueError('Missing action_id for some rows after merge')

# Avoid column suffix collisions by removing any prior derived columns before merging
for c in ['x_start_norm','y_start_norm','x_end_norm','y_end_norm','success']:
    if c in df.columns:
        df = df.drop(columns=[c])
    if f'{c}_x' in df.columns or f'{c}_y' in df.columns:
        df = df.drop(columns=[col for col in [f'{c}_x', f'{c}_y'] if col in df.columns])

clean_cols = [
    'match_id','event_id','action_id',
    'x_start_norm','y_start_norm','x_end_norm','y_end_norm','success',
]
df_clean = pd.read_parquet(ACTIONS_CLEAN, columns=clean_cols)
df_clean = df_clean[df_clean['match_id'].isin(match_ids)].copy()

left_dups = int(df.duplicated(['match_id','event_id','action_id']).sum())
right_dups = int(df_clean.duplicated(['match_id','event_id','action_id']).sum())
print('Duplicate (match_id,event_id,action_id) keys - left:', left_dups, 'right:', right_dups)

# Use one row per (match_id,event_id,action_id) for this analysis
# (the upstream table may contain multiple rows per event_id due to extra associated records)
df = df.drop_duplicates(['match_id','event_id','action_id'])
df_clean = df_clean.drop_duplicates(['match_id','event_id','action_id'])

df = df.merge(df_clean, on=['match_id','event_id','action_id'], how='left', validate='one_to_one')

print('Missing normalized coords fraction:', float(df['x_start_norm'].isna().mean()))
print('Missing success fraction:', float(df['success'].isna().mean()))

display(df.head())


Duplicate (match_id,event_id,action_id) keys - left: 0 right: 0
Missing normalized coords fraction: 0.0
Missing success fraction: 0.0


,match_id,period,frame_start,frame_end,event_id,team_id,player_id,event_type,end_type,pass_outcome,x_start,y_start,x_end,y_end,team_in_possession_phase_type,phase_index,action_id,x_start_norm,y_start_norm,x_end_norm,y_end_norm,success
0,1650385,1,52,52,8_0,48,11438,player_possession,pass,successful,0.37,-0.51,0.37,-0.51,create,0,0,52.873558,33.49,52.873558,33.49,1
1,1650385,1,62,71,8_1,48,13068,player_possession,pass,successful,-9.60,-0.26,-9.72,-1.24,create,0,1,42.807692,33.74,42.686538,32.76,1
2,1650385,1,87,124,8_2,48,4504,player_possession,pass,successful,-25.30,-6.43,-22.56,-20.46,create,0,2,26.956731,27.57,29.723077,13.54,1
3,1650385,1,87,124,8_2,48,4504,player_possession,pass,successful,-25.30,-6.43,-22.56,-20.46,create,0,337390,26.956731,27.57,29.723077,13.54,1
4,1650385,1,160,182,8_4,31,70672,player_possession,pass,successful,-8.05,23.63,-13.67,32.22,chaotic,1,3,44.372596,57.63,38.698558,66.22,1


In [9]:
# 1) WHERE DOES THE OPPONENT REGAIN THE BALL?
#
# For each row i, find the next player_possession event within the same (match, period)
# where the team_id is different. That next row is our best-available proxy for: 
# - opponent regain location (x_start_norm, y_start_norm)
# - opponent regain time gap (frames between end of i and start of that opponent possession)

df = df.sort_values(['match_id','period','frame_start','frame_end']).reset_index(drop=True)

# next row info
g = df.groupby(['match_id','period'], sort=False)
df['next_team_id'] = g['team_id'].shift(-1)
df['next_frame_start'] = g['frame_start'].shift(-1)
df['next_x_start_norm'] = g['x_start_norm'].shift(-1)
df['next_y_start_norm'] = g['y_start_norm'].shift(-1)
df['next_phase_type'] = g['team_in_possession_phase_type'].shift(-1)

df['team_switch_next'] = df['next_team_id'].notna() & (df['next_team_id'] != df['team_id'])
df['gap_to_next_frames'] = df['next_frame_start'] - df['frame_end']

# Opponent regain proxy (only defined when team switches next)
df['opp_regain_x_norm'] = np.where(df['team_switch_next'], df['next_x_start_norm'], np.nan)
df['opp_regain_y_norm'] = np.where(df['team_switch_next'], df['next_y_start_norm'], np.nan)
df['opp_regain_gap_frames'] = np.where(df['team_switch_next'], df['gap_to_next_frames'], np.nan)
df['opp_regain_phase_type'] = np.where(df['team_switch_next'], df['next_phase_type'], None)

print('Team switches next (sample):', int(df['team_switch_next'].sum()), 'of', len(df))
display(df.loc[df['team_switch_next'], [
    'match_id','period','event_id','team_id','end_type','pass_outcome','success',
    'x_start_norm','y_start_norm','frame_end',
    'next_team_id','next_phase_type','opp_regain_x_norm','opp_regain_y_norm','opp_regain_gap_frames'
]].head(12))

print('opp_regain_gap_frames quantiles (team switches):')
display(pd.Series(df.loc[df['team_switch_next'], 'opp_regain_gap_frames']).quantile([0.5,0.75,0.9,0.95,0.99]))
print('opp_regain_phase_type counts (team switches):')
display(df.loc[df['team_switch_next'], 'opp_regain_phase_type'].value_counts(dropna=False).head(15))

Team switches next (sample): 554 of 3918


,match_id,period,event_id,team_id,end_type,pass_outcome,success,x_start_norm,y_start_norm,frame_end,next_team_id,next_phase_type,opp_regain_x_norm,opp_regain_y_norm,opp_regain_gap_frames
4,1650385,1,8_3,48,possession_loss,None,0,65.897596,4.45,152,31.0,chaotic,44.372596,57.63,8.0
8,1650385,1,8_7,31,pass,successful,1,19.374519,51.52,655,48.0,chaotic,69.835096,21.04,31.0
9,1650385,1,8_8,48,pass,successful,1,69.835096,21.04,686,31.0,create,26.502404,34.56,17.0
62,1650385,1,8_41,31,pass,unsuccessful,0,77.831250,52.86,1850,48.0,chaotic,21.878365,16.78,10.0
82,1650385,1,8_55,48,pass,unsuccessful,1,39.667788,28.79,2338,31.0,chaotic,43.564904,39.37,10.0
83,1650385,1,8_56,31,pass,unsuccessful,0,43.564904,39.37,2348,48.0,create,41.394231,38.74,20.0
97,1650385,1,8_65,48,pass,successful,1,71.935096,3.03,2812,31.0,transition,4.018269,29.26,614.0
114,1650385,1,8_75,31,pass,unsuccessful,0,70.683173,66.96,3847,48.0,transition,16.275000,13.99,10.0
118,1650385,1,8_78,48,pass,unsuccessful,1,30.773077,4.57,3994,31.0,build_up,8.268750,39.99,13.0
124,1650385,1,8_82,31,possession_loss,None,0,21.514904,54.90,4084,48.0,finish,65.513942,9.38,31.0


opp_regain_gap_frames quantiles (team switches):


0.50     25.00
0.75    116.00
0.90    344.40
0.95    577.60
0.99    977.04
Name: opp_regain_gap_frames, dtype: float64

opp_regain_phase_type counts (team switches):


opp_regain_phase_type
chaotic        163
create         113
build_up       107
finish          52
transition      36
disruption      32
set_play        18
direct          18
quick_break     15
Name: count, dtype: int64

In [10]:
# 2) TURNOVER VS RESTART (HEURISTICS)
#
# The underlying SkillCorner dynamic player-possession table does not expose explicit restart types
# like 'throw-in'/'corner'/'goal kick'. We therefore test restart-like heuristics based on:
#   A) a large time gap between the end of our action and the start of the opponent possession
#   B) whether the opponent starts in a 'set_play' phase type (covers corners, free-kicks, long throws)
#
# NOTE: Goal-kicks and many throw-ins may be labelled as build_up/create rather than set_play.

# Define failure across all action types (in this project pipeline, success is already derived in actions_clean)
df['is_failure'] = (df['success'] == 0)

fails = df[df['is_failure'] & df['team_switch_next']].copy()
print('Failures that immediately switch possession (sample):', len(fails))

fails['next_is_set_play'] = (fails['opp_regain_phase_type'] == 'set_play')

def summarize_threshold(thresh: int) -> dict:
    restart_like = fails['opp_regain_gap_frames'] >= thresh
    return {
        'gap_thresh_frames': thresh,
        'restart_like_count': int(restart_like.sum()),
        'restart_like_rate': float(restart_like.mean()),
        'set_play_rate': float(fails['next_is_set_play'].mean()),
        'overlap_count': int((restart_like & fails['next_is_set_play']).sum()),
        'precision_vs_set_play': float(((restart_like & fails['next_is_set_play']).sum() / max(1, restart_like.sum()))),
        'recall_vs_set_play': float(((restart_like & fails['next_is_set_play']).sum() / max(1, fails['next_is_set_play'].sum()))),
    }

thresholds = [25, 50, 100, 200, 300, 500]
summary = pd.DataFrame([summarize_threshold(t) for t in thresholds])
display(summary)

# Pick a working threshold for 'restart-like' (feel free to tweak)
RESTART_GAP_FRAMES = 200
fails['restart_like_gap'] = fails['opp_regain_gap_frames'] >= RESTART_GAP_FRAMES
fails['restart_like'] = fails['restart_like_gap'] | fails['next_is_set_play']

print('restart_like (combined) rate:', fails['restart_like'].mean())
display(fails[['opp_regain_gap_frames','opp_regain_phase_type','restart_like_gap','next_is_set_play','restart_like']].value_counts().head(12))

# Who gets the restart? In this dataset, best proxy is simply the team of the next possession start
fails['restart_team_id'] = fails['next_team_id']

print('Example failures classified as restart_like:')
display(fails.loc[fails['restart_like'], [
    'match_id','period','event_id','team_id','end_type','pass_outcome','x_start_norm','y_start_norm',
    'restart_team_id','opp_regain_phase_type','opp_regain_x_norm','opp_regain_y_norm','opp_regain_gap_frames'
]].head(15))

Failures that immediately switch possession (sample): 312


,gap_thresh_frames,restart_like_count,restart_like_rate,set_play_rate,overlap_count,precision_vs_set_play,recall_vs_set_play
0,25,152,0.487179,0.051282,6,0.039474,0.3750
1,50,93,0.298077,0.051282,3,0.032258,0.1875
2,100,82,0.262821,0.051282,3,0.036585,0.1875
3,200,61,0.195513,0.051282,3,0.049180,0.1875
4,300,37,0.118590,0.051282,2,0.054054,0.1250
5,500,16,0.051282,0.051282,1,0.062500,0.0625


restart_like (combined) rate: 0.23717948717948717


opp_regain_gap_frames  opp_regain_phase_type  restart_like_gap  next_is_set_play  restart_like
13.0                   chaotic                False             False             False           6
20.0                   chaotic                False             False             False           4
14.0                   chaotic                False             False             False           4
42.0                   chaotic                False             False             False           4
15.0                   create                 False             False             False           4
3.0                    create                 False             False             False           4
19.0                   chaotic                False             False             False           4
22.0                   chaotic                False             False             False           3
4.0                    chaotic                False             False             False           3
11.0 

Example failures classified as restart_like:


,match_id,period,event_id,team_id,end_type,pass_outcome,x_start_norm,y_start_norm,restart_team_id,opp_regain_phase_type,opp_regain_x_norm,opp_regain_y_norm,opp_regain_gap_frames
167,1650385,1,8_114,31,possession_loss,None,64.645673,5.28,48.0,create,49.521635,26.63,650.0
233,1650385,1,8_163,31,foul_suffered,None,44.352404,51.78,48.0,build_up,14.720192,51.36,822.0
246,1650385,1,8_173,48,possession_loss,None,42.666346,16.97,31.0,build_up,13.629808,15.12,227.0
384,1650385,1,8_270,31,shot,None,93.086538,31.00,48.0,direct,12.711058,53.10,255.0
623,1650385,1,8_430,48,clearance,None,20.434615,61.50,31.0,finish,93.853846,2.10,238.0
676,1650385,1,8_467,48,pass,unsuccessful,9.278365,31.15,31.0,set_play,82.647115,42.32,23.0
688,1650385,1,8_478,31,shot,None,85.302404,31.59,48.0,build_up,4.008173,35.00,254.0
698,1650385,1,8_487,48,possession_loss,None,72.853846,7.28,31.0,build_up,8.076923,41.55,246.0
805,1650385,2,8_564,31,foul_suffered,None,8.682692,54.32,48.0,chaotic,39.687981,14.52,392.0
825,1650385,2,8_578,48,clearance,None,24.543750,63.11,31.0,finish,81.677885,14.87,211.0


In [11]:
# Optional: spatial sanity checks
# If restarts (throw-ins / goal kicks / corners) were explicitly present as possession starts,
# we would expect many restart-like opponent regains to cluster near touchlines or the defending goal line.

# Estimate pitch bounds from normalized on-ball positions in the sample
hx = df['x_start_norm'].abs().quantile(0.999)
hy = df['y_start_norm'].abs().quantile(0.999)
print('Estimated |x| bound, |y| bound (norm):', hx, hy)

eps = 1.0
near_touch = fails['opp_regain_y_norm'].abs() > (hy - eps)
near_goal_line = fails['opp_regain_x_norm'].abs() > (hx - eps)

tab = pd.DataFrame({
    'near_touchline': near_touch,
    'near_goalline': near_goal_line,
    'restart_like': fails['restart_like'],
}).groupby(['restart_like','near_touchline','near_goalline']).size().reset_index(name='count')
display(tab.sort_values('count', ascending=False).head(20))

print('Near touchline rate among restart_like:', float(near_touch[fails['restart_like']].mean()))
print('Near goalline rate among restart_like:', float(near_goal_line[fails['restart_like']].mean()))

# If these rates stay very low, it likely means the *restart taker action itself*
# is not represented as a player_possession start in this dataset (e.g., throw-in is not a player possession).

Estimated |x| bound, |y| bound (norm): 103.35847644230779 68.0


,restart_like,near_touchline,near_goalline,count
0,False,False,False,236
2,True,False,False,74
1,False,False,True,2


Near touchline rate among restart_like: 0.0
Near goalline rate among restart_like: 0.0
